## Pretraining

### Load all allowed libraries

In [ ]:
import numpy as np
from PIL import Image
import pandas as pd
import sklearn
import scipy
import seaborn as sns
import torch
import torchinfo
import torchvision
from tqdm import tqdm

import matplotlib.pyplot as plt

### Define data transfromation function, and data augmentation

In [2]:
from torchvision.transforms import v2

# Training data transforms and augmentation
training_tfms_and_agmt = v2.Compose([
    v2.ToImage(), # this turns it into a tensor

    v2.RandomRotation(degrees=15), # for realistic pet location variations 
    v2.RandomAffine(degrees=10, translate=(0.1, 0.1)), # for better position accuracy
    v2.RandomHorizontalFlip(p=0.5), # it is still the same cat if flip like this but double data

    v2.RandomResizedCrop((224, 224), scale=(0.8, 1.0), antialias=True), # 224 * 224 seems to be what everyone doing
    v2.ColorJitter(brightness=0.3, contrast=0.2, saturation=0.2, hue=0.05), # this is for pet photographic varience

    v2.ToDtype(torch.float32, scale=True), # apply the min-max normalization and scale is for 0-255 to num between 0 and 1
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]), # apply the standardization for the z-scores things
])

valutation_tfms = v2.Compose([
    v2.ToImage(),
    v2.Resize(256, antialias=True),
    v2.CenterCrop(224),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # this normalization values are for now, can be better
])

### Load dataset and create custom dataset

In [3]:
from torch.utils.data import Dataset, random_split
from torchvision import datasets

class TransformWrapper(Dataset):
    def __init__(self, subset, transform=None):
        super().__init__() # inherit the Dataset init method just in case even though I think there is nothing
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

raw_data = datasets.OxfordIIITPet(root="/shared/storage/cs/studentscratch/cqh514", split='trainval', download=True)

### Randomly split but make the split constant accross different runes, in a 97.5:2.5 training, valuation configuration

In [4]:
# no dataset splitting, use all for training
raw_training = raw_data

### Inject the transforms to the raw subset that is created with the split we just made

In [5]:
# Wrap the raw subsets to inject the transforms on the fly
training_dataset = TransformWrapper(raw_training, transform=training_tfms_and_agmt)

### Preparing data with dataloaders, don't shuffle the valuation then no waste cpu

In [6]:
from torch.utils.data import DataLoader
training_loader = DataLoader(training_dataset, batch_size=32, shuffle=True, num_workers=10)

### Define the first 1080ti in the server for training

# Training

### Stuff to change to run on different GPUs

In [ ]:
model_save_path = "/shared/storage/cs/studentscratch/cqh514/training_loop_results/gpu1_2.pth"
device = torch.device("cuda:1")

### Define the neural network by subclassing the nn.Module

In [8]:
import torch.nn as nn

class SEResBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
        )
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.block(x)
        scale = self.se(out).view(x.size(0), -1, 1, 1)
        return self.relu(x + out * scale)

class OxfordPetClassifierCNN(nn.Module):
    def __init__(self, num_classes=37):
        super().__init__()
        
        # Convolutional Layers, adding more then 5 seems to decrease accuracy (this is before the resblock is added)
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=7, padding=3),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            SEResBlock(64),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            SEResBlock(128),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            SEResBlock(256),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            SEResBlock(512),
            nn.MaxPool2d(2, 2),
        )
        
        # Only one linear Layer with AdaptiveAvgPool2d instead of multiple flatten 
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        output = self.classifier(x) # logits is the name for the numbers outputed before softmax turns it into probabilities
        return output

model = OxfordPetClassifierCNN().to(device)
print(model)

OxfordPetClassifierCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): SEResBlock(
      (block): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (se): Sequential(
        (0): AdaptiveAvgPool2d(output_size=1)
        (1): Flat

### Set loss function and its optimizer

In [9]:
# use the cross entropy loss with label smoothing, instead of like 1 or 0 for classification, make it less definitive
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Adam optimizer seems to be the best and industry standard
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

### Run the training loop

In [10]:
import time

EPOCHS = 30

# Tell the scheduler exactly how many total steps there are
total_steps = len(training_loader) * EPOCHS

# The one cycle Scheduler, basicly peak learning rate at epoch 15 and slow down by epoch 30, so it will be best for my 30 epoch limit
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-3, # peak learning rate
    total_steps=total_steps, # this is basicly 30
    pct_start=0.3, # 30% is warm up 
    div_factor=25.0, # div by this and you got the starting learning rate
    final_div_factor=1000.0 # div by this and you got ending learning rate
)

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print("-" * 15)
    start_time = time.time() # for seeing how long a epoch takes and might just use another gpu
    
    # training starts here
    model.train() # this basicly activates the dropout but i removed it now just normalize based on current batch's mean/varience
    running_loss = 0.0 # rset stuff after each epoch
    correct_train = 0
    total_train = 0
    
    for inputs, labels in tqdm(training_loader, desc="Training", leave=False): # the tqdm is the progress bar thing
        inputs, labels = inputs.to(device), labels.to(device) # move data to the vram so it faster
        
        optimizer.zero_grad() # clear gradient from last batch
        outputs = model(inputs) # forward the data
        loss = criterion(outputs, labels) # run loss function
        loss.backward() # back probagation
        
        optimizer.step() # update the weights 
        
        scheduler.step() # update the learning rate for the one cycle lr
        
        running_loss += loss.item() * inputs.size(0) # loss per epoch
        _, predicted = torch.max(outputs, 1) # get the prediciton no need the score
        total_train += labels.size(0) # get total images
        correct_train += (predicted == labels).sum().item() # count correct ones
        
    train_acc = correct_train / total_train # the training accuracy
    current_lr = optimizer.param_groups[0]['lr'] # current learning rate
    
    print(f"LR: {current_lr:.5f} | Train Acc: {train_acc * 100:.2f}% | Time: {time.time() - start_time:.0f}s")
    
# save the model after all epochs finish
torch.save(model.state_dict(), model_save_path)
print(f"🌟 Saved final model at epoch {EPOCHS} with Train Acc: {train_acc * 100:.2f}%")

print(f"\\nTraining done!")

Epoch 1/30
---------------


Training:   0%|                                                                                                               | 0/115 [00:00<?, ?it/s]

LR: 0.00021 | Train Acc: 6.90% | Time: 22s
Epoch 2/30
---------------


LR: 0.00046 | Train Acc: 10.14% | Time: 22s
Epoch 3/30
---------------


LR: 0.00084 | Train Acc: 11.93% | Time: 20s
Epoch 4/30
---------------


...

LR: 0.00026 | Train Acc: 93.18% | Time: 19s
Epoch 27/30
---------------


LR: 0.00015 | Train Acc: 94.32% | Time: 19s
Epoch 28/30
---------------


LR: 0.00007 | Train Acc: 95.90% | Time: 19s
Epoch 29/30
---------------


LR: 0.00002 | Train Acc: 96.14% | Time: 21s
Epoch 30/30
---------------


LR: 0.00000 | Train Acc: 95.84% | Time: 20s
🌟 Saved final model at epoch 30 with Train Acc: 95.84%
\nTraining done!


### Testing transform (same as the valuation transform)

In [11]:
from torchvision.transforms import v2


test_tfms = v2.Compose([ # this needs to be same as the valuation transform
    v2.ToImage(),
    v2.Resize(256, antialias=True),
    v2.CenterCrop(224),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [12]:
raw_test_data = datasets.OxfordIIITPet(root="/shared/storage/cs/studentscratch/cqh514", split='test', download=True)
test_dataset = TransformWrapper(raw_test_data, transform=test_tfms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=10)

In [13]:
model = OxfordPetClassifierCNN(num_classes=37).to(device)
model.load_state_dict(torch.load(model_save_path))

# set to evaulation mode
model.eval()

correct_test = 0
total_test = 0

print("Starting evaluation on the test dataset...")

# disable gradient calculations for testing, basicly same code as the validation phase in the training loop
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

test_acc = correct_test / total_test
print(f"\nFinal Test Accuracy: {test_acc * 100:.2f}%")

Starting evaluation on the test dataset...


Testing: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 115/115 [00:08<00:00, 13.87it/s]


Final Test Accuracy: 69.20%
